# Lab 3.2 · An agent SDK against your own loop

Day 3 · Session 2 · 25 min. Agent SDKs package the loop, tools, sessions, guardrails and tracing. In this lab you
rebuild Bosla on one SDK (the OpenAI Agents SDK, pointed at a free OpenRouter model), run the same ten tasks, and
compare it with the reference loop on three things: code size, what the trace shows, and what you control.

**Reference state.** `bosla.py` (the own-loop agent), `port.py`, the ten Day 1 tasks. Nothing from Lab 3.1.

**Tasks**
- [ ] 1 · Bosla on the SDK: model, tools, agent (7')
- [ ] 2 · The same ten tasks, graded the same way (6')
- [ ] 3 · Traces and control, side by side (7')
- [ ] 4 · The comparison table (5')
- [ ] ✱ Stretch · The write gate in the SDK

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_URL = "https://github.com/SDAIAAcademy/agentic-ai.git"
    if not Path("/content/agentic-ai").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/agentic-ai"], check=True)
    os.chdir("/content/agentic-ai/labs")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
else:
    while not Path("agentkit.py").exists() and Path.cwd() != Path.cwd().parent:
        os.chdir("..")  # solutions/ notebooks run from labs/
sys.path.insert(0, os.getcwd())

import agentkit as K
print("provider:", K.PROVIDER, "| model:", K.MODEL)

In [ ]:
import asyncio
import inspect
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
from agents import Agent, OpenAIChatCompletionsModel, Runner, function_tool, set_tracing_disabled
from openai import AsyncOpenAI

import bosla
import port
import porttasks

set_tracing_disabled(True)  # the SDK exports traces to OpenAI's dashboard by default; we keep them local
TASKS = porttasks.load_tasks("day1_tasks")


def run_sync(coro):
    """Run a coroutine from a notebook (which already has an event loop) or from a script."""
    with ThreadPoolExecutor(1) as pool:
        return pool.submit(asyncio.run, coro).result()

## 1 · Bosla on the SDK (7')

The SDK speaks the Responses API to OpenAI by default. For any other provider, wrap an `AsyncOpenAI` client in
`OpenAIChatCompletionsModel`. `function_tool` reads the same type hints and docstrings as `K.tool_schema`.

✏️ **Exercise** · Write `make_sdk_agent(p)`: the model for OpenRouter, every port tool through `function_tool`,
and an `Agent` named "Bosla" with the reference prompt plus the context file as instructions.

In [ ]:
def make_sdk_agent(p: port.Port) -> Agent:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")


p = port.Port(K.lab_dir("D3_2/warmup", reset=True))
result = run_sync(Runner.run(make_sdk_agent(p), "Berth hire for Najm Falcon for 2 days?", max_turns=8))
print(result.final_output)
for item in result.new_items:
    print(f"  {type(item).__name__:22s}", getattr(item.raw_item, "name", "") or "")

## 2 · The same ten tasks (6')

The SDK reports tool calls as run items. To grade with `porttasks.grade` you need `(name, args)` pairs.

✏️ **Exercise** · For each task: a fresh port, the SDK agent, `Runner.run(..., max_turns=8)`, the tool calls from
`result.new_items` (items whose type is `ToolCallItem`: name and JSON arguments on `raw_item`), the grade. An
exception from the SDK (for example `MaxTurnsExceeded`) is a failed task with the exception's name as `why`.

In [ ]:
import json

rows = []
for t in TASKS:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")
sdk = pd.DataFrame(rows)
print(sdk.to_string(index=False))
K.check(len(sdk) == 10, "ten tasks on the SDK")

🔍 T04 says "do not book yet" and T09 uses the wrong agent code. The SDK agent has no write gate: which of the two
does the tool's own check still protect, and which depends on the model alone?

## 3 · Traces and control (7')

✏️ **Exercise** · Fill `control`: for each harness part of `bosla.PARTS` (except `verify`), how you would get it
on the SDK, in a few words, or "not built in". Then count the lines of the two agents' code: `bosla.run` plus
`agentkit.run_agent` against `make_sdk_agent` plus the run call.

In [ ]:
# ✏️ YOUR CODE HERE
raise NotImplementedError("✏️ Exercise: write your solution here")
print(pd.DataFrame(control.items(), columns=["part", "on the SDK"]).to_string(index=False))
print("lines: own loop", own_lines, "| SDK agent", sdk_lines)

## 4 · The comparison table (5')

✏️ **Exercise** · Write `table`: rows `code size`, `trace`, `control`, `passed tasks`; columns `own loop` and
`SDK`. Use your numbers, and one line of judgement per row. This is the table the capstone README asks for when
you choose between an SDK and your own harness.

In [ ]:
# ✏️ YOUR CODE HERE
raise NotImplementedError("✏️ Exercise: write your solution here")
print(table.to_string(index=False))

## ✱ Stretch · The write gate in the SDK

Read the SDK's documentation for tool approvals or tool guardrails, and add a gate that refuses
`create_booking`, `amend_booking` and `cancel_booking` unless the input says "I confirm". Run T04 again.

## Checkpoint

1. Name one thing you lost and one thing you gained by moving to the SDK.
2. The SDK raises when `max_turns` is reached. What does your evaluation code have to do about that?
3. Where, in the SDK's run, would you look for the equivalent of your trace's `guard` events?